# Plant Disease Classification — Classical ML Ensemble

**Goal:** Classify plant leaf images into disease categories using three classical ML models (KNN, SVM, Random Forest) combined via a weighted soft-voting ensemble.

**Key challenge addressed:** The *lab-to-field performance gap* — models trained only on clean, lab-quality PlantVillage images often fail on real-world PlantDoc images taken under variable lighting, angles, and backgrounds. We address this through:
- Mixed dataset training (PlantVillage + PlantDoc)
- Robust preprocessing (CLAHE, Gray-World, leaf segmentation)
- Multi-family feature engineering (colour + texture + shape)

---

## Notebook Structure
| Step | Description |
|------|-------------|
| 0 | Environment setup & imports |
| 1 | Data discovery & split creation |
| 2 | Feature extraction (with caching) |
| 3 | Train KNN, SVM, Random Forest |
| 4 | Build weighted ensemble |
| 5 | Evaluate on full test set |
| 6 | Evaluate on field-only (PlantDoc) subset |
| 7 | Visualise results |
| 8 | Save final models |

---
## Step 0 — Environment Setup & Imports

We add the project root to `sys.path` so Python can find the `src/` package regardless of how the notebook is launched.

In [ ]:
import sys
import os
import pathlib

# ── Locate project root and add to path ───────────────────────────────────
# The notebook lives in notebooks/, so the project root is one level up.
PROJECT_ROOT = pathlib.Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]}")

In [ ]:
# ── Standard library & third-party ────────────────────────────────────────
import warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")

# ── Project modules ───────────────────────────────────────────────────────
from src.utils              import (
    build_full_dataframe, make_splits, load_splits,
    save_features, load_features, features_cached,
    print_class_distribution, print_source_distribution,
    summarise_splits, set_seed, resolve_path
)
from src.feature_extraction import (
    extract_features_from_dataset, get_feature_names
)
from src.models    import train_all_models, load_model, get_val_accuracy
from src.ensemble  import build_ensemble, WeightedEnsembleClassifier
from src.evaluation import (
    evaluate_all, evaluate_model, print_metrics,
    plot_confusion_matrix, plot_per_class_f1, plot_model_comparison
)

# ── Reproducibility ───────────────────────────────────────────────────────
set_seed(42)

# ── Matplotlib display settings ───────────────────────────────────────────
%matplotlib inline
plt.rcParams.update({
    "figure.dpi"     : 120,
    "axes.titlesize" : 13,
    "axes.labelsize" : 11,
})

print("All imports successful.")

In [ ]:
# ── Directory configuration ───────────────────────────────────────────────
DATA_DIR         = str(PROJECT_ROOT / "data")
PLANTVILLAGE_DIR = str(PROJECT_ROOT / "data" / "plantvillage")
PLANTDOC_DIR     = str(PROJECT_ROOT / "data" / "plantdoc")
SPLITS_DIR       = str(PROJECT_ROOT / "data" / "splits")
MODELS_DIR       = str(PROJECT_ROOT / "models")
REPORTS_DIR      = str(PROJECT_ROOT / "reports")
FIGURES_DIR      = str(PROJECT_ROOT / "reports" / "figures")

# Feature cache paths
CACHE = {
    "train" : str(PROJECT_ROOT / "data" / "splits" / "train_features"),
    "val"   : str(PROJECT_ROOT / "data" / "splits" / "val_features"),
    "test"  : str(PROJECT_ROOT / "data" / "splits" / "test_features"),
}

# Ensure output directories exist
for d in [MODELS_DIR, REPORTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print("Directories configured.")

---
## Step 1 — Data Discovery & Stratified Split

We scan both dataset folders and build a unified DataFrame, then create a **70 / 15 / 15** stratified split so every class is proportionally represented in train, val, and test.

> **Note:** If the `data/splits/*.csv` files already exist, we skip scanning and load them directly. This makes repeated runs fast.

In [ ]:
TRAIN_CSV = os.path.join(SPLITS_DIR, "train.csv")

if os.path.exists(TRAIN_CSV):
    # ── Load existing splits ──────────────────────────────────────────────
    print("Found existing splits — loading from CSV files.")
    train_df, val_df, test_df = load_splits(SPLITS_DIR)
else:
    # ── Discover images and create splits ────────────────────────────────
    print("No splits found — scanning dataset directories …")
    full_df = build_full_dataframe(PLANTVILLAGE_DIR, PLANTDOC_DIR)
    train_df, val_df, test_df = make_splits(
        full_df, splits_dir=SPLITS_DIR, seed=42
    )

print("\nSplit summary:")
summarise_splits(train_df, val_df, test_df)

In [ ]:
# ── Inspect class and source distributions ────────────────────────────────
print_class_distribution(train_df, "Training Set — Class Distribution")
print_source_distribution(train_df)

In [ ]:
# ── Visualise class balance ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (df, title) in zip(
    axes,
    [(train_df, "Train"), (val_df, "Validation"), (test_df, "Test")]
):
    counts = df["label"].value_counts().sort_values(ascending=True)
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(counts)))
    ax.barh(counts.index, counts.values, color=colors)
    ax.set_title(f"{title} ({len(df):,} samples)")
    ax.set_xlabel("Image Count")
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle("Class Distribution Across Splits", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "class_distribution.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/figures/class_distribution.png")

---
## Step 2 — Feature Extraction

Each image is passed through the full preprocessing pipeline, then three families of features are extracted:

| Family  | Features | Description |
|---------|----------|-------------|
| Color   | 20       | Mean/std of R,G,B,H,S,V channels + vegetation indices (NDI, ExG, G/R, G/B) |
| Texture | 30       | GLCM properties (contrast, correlation, energy, homogeneity) + 26-bin LBP histogram |
| Shape   | 6        | Area, perimeter, compactness, eccentricity, aspect ratio, solidity |
| **Total** | **56** | Combined feature vector per image |

Results are cached as `.npz` files so this step only runs once.

In [ ]:
# ── Extract or load cached features ───────────────────────────────────────

def get_features(df: pd.DataFrame, cache_path: str,
                 split_name: str = ""):
    """Load features from cache if available, else extract and cache."""
    if features_cached(cache_path):
        print(f"  [{split_name}] Loading from cache: {cache_path}.npz")
        X, y = load_features(cache_path)
    else:
        print(f"  [{split_name}] Extracting features for {len(df)} images …")
        X, y, skipped = extract_features_from_dataset(df, show_progress=True)
        save_features(cache_path, X, y)
        if skipped:
            print(f"  [{split_name}] Skipped {len(skipped)} corrupted images.")
    return X, y


print("=" * 55)
print("  Feature Extraction")
print("=" * 55)

X_train, y_train = get_features(train_df, CACHE["train"], "TRAIN")
X_val,   y_val   = get_features(val_df,   CACHE["val"],   "VAL"  )
X_test,  y_test  = get_features(test_df,  CACHE["test"],  "TEST" )

print(f"\nFeature matrix shapes:")
print(f"  X_train : {X_train.shape}")
print(f"  X_val   : {X_val.shape}")
print(f"  X_test  : {X_test.shape}")

In [ ]:
# ── Feature statistics: check for NaN / Inf ───────────────────────────────
for name, X in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    n_nan = np.isnan(X).sum()
    n_inf = np.isinf(X).sum()
    print(f"  {name:<6s}: shape={X.shape}, NaN={n_nan}, Inf={n_inf}, "
          f"mean={X.mean():.4f}, std={X.std():.4f}")

In [ ]:
# ── Feature importance preview: variance per feature ──────────────────────
feat_names  = get_feature_names()   # 56 names
feat_vars   = X_train.var(axis=0)

top_idx  = np.argsort(feat_vars)[-20:][::-1]   # top-20 by variance
fig, ax  = plt.subplots(figsize=(12, 4))
ax.bar([feat_names[i] for i in top_idx],
       feat_vars[top_idx],
       color="steelblue", edgecolor="white")
ax.set_title("Top-20 Features by Variance (Training Set)")
ax.set_ylabel("Variance")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "feature_variance.png"),
            dpi=150, bbox_inches="tight")
plt.show()

---
## Step 3 — Train Individual Models

Each model is trained with **GridSearchCV** (5-fold stratified CV) to find the best hyperparameters. All models include a `StandardScaler` inside their Pipeline so scaling is automatic at inference time.

| Model | Tuned hyperparameters |
|-------|----------------------|
| KNN   | `n_neighbors` ∈ {3,5,7,9,11}, `metric` ∈ {euclidean, manhattan} |
| SVM   | `C` ∈ {0.1,1,10,100}, `gamma` ∈ {scale, auto, 0.001, 0.01} |
| RF    | `max_depth` ∈ {None,10,20,30}, `min_samples_leaf` ∈ {1,2,4}, `max_features` ∈ {sqrt, log2} |

> ⏱ **Estimated training time:** 5–30 minutes depending on dataset size and CPU cores available. `n_jobs=-1` uses all cores.

In [ ]:
# ── Train (or reload if already saved) ────────────────────────────────────
MODEL_PATHS = {
    "knn" : os.path.join(MODELS_DIR, "knn_model.pkl"),
    "svm" : os.path.join(MODELS_DIR, "svm_model.pkl"),
    "rf"  : os.path.join(MODELS_DIR, "rf_model.pkl"),
}

all_saved = all(os.path.exists(p) for p in MODEL_PATHS.values())

if all_saved:
    print("Saved models found — loading from disk.")
    model_results = {}
    for name, path in MODEL_PATHS.items():
        m       = load_model(path)
        val_acc = get_val_accuracy(m, X_val, y_val)
        model_results[name] = {"model": m, "val_acc": val_acc, "save_path": path}
        print(f"  {name.upper():<4s}  val_acc = {val_acc:.4f}")
else:
    print("Training models from scratch …")
    model_results = train_all_models(
        X_train, y_train,
        X_val,   y_val,
        models_dir = MODELS_DIR,
        cv_folds   = 5
    )

In [ ]:
# ── Summary of individual model validation accuracies ─────────────────────
val_summary = pd.DataFrame([
    {"Model": name.upper(), "Val Accuracy": info["val_acc"]}
    for name, info in model_results.items()
])
val_summary["Val Accuracy"] = val_summary["Val Accuracy"].round(4)
display(val_summary.style.bar(
    subset=["Val Accuracy"], color="#5bc0de"
).format({"Val Accuracy": "{:.4f}"}))

---
## Step 4 — Build Weighted Ensemble

The ensemble combines KNN, SVM, and RF using **weighted soft voting**:

$$P_{\text{ensemble}}(c \mid x) = \frac{\sum_i w_i \cdot P_i(c \mid x)}{\sum_i w_i}$$

Each model's weight $w_i$ equals its **validation accuracy** — so the best-performing base model has the most influence on the final prediction.

In [ ]:
ENSEMBLE_PATH = os.path.join(MODELS_DIR, "ensemble.pkl")

if os.path.exists(ENSEMBLE_PATH):
    print("Loading saved ensemble …")
    ensemble = WeightedEnsembleClassifier.load(ENSEMBLE_PATH)
else:
    print("Building ensemble …")
    ensemble = build_ensemble(
        model_results,
        X_train, y_train,
        save_path = ENSEMBLE_PATH
    )

# Quick sanity check on validation set
ens_val_acc = ensemble.score(X_val, y_val)
print(f"\nEnsemble validation accuracy : {ens_val_acc:.4f}")

In [ ]:
# ── Weight visualisation ──────────────────────────────────────────────────
weights_norm = ensemble.weights_norm_
model_names  = ["KNN", "SVM", "RF"]

fig, ax = plt.subplots(figsize=(7, 4))
bars    = ax.bar(model_names, weights_norm,
                 color=["#5bc0de", "#d9534f", "#5cb85c"], edgecolor="white", width=0.5)
for bar, w in zip(bars, weights_norm):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f"{w:.4f}", ha="center", va="bottom", fontsize=11)
ax.set_ylim(0, max(weights_norm) * 1.25)
ax.set_ylabel("Normalised Weight")
ax.set_title("Ensemble Weights per Base Model")
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "ensemble_weights.png"),
            dpi=150, bbox_inches="tight")
plt.show()

---
## Step 5 — Evaluate on Full Test Set

We evaluate all four models (KNN, SVM, RF, Ensemble) on the **full test set** (both PlantVillage + PlantDoc images combined).

In [ ]:
# ── Build models dict for evaluate_all ────────────────────────────────────
models_dict = {
    "knn"      : model_results["knn"]["model"],
    "svm"      : model_results["svm"]["model"],
    "rf"       : model_results["rf"]["model"],
    "ensemble" : ensemble,
}

# ── Run full evaluation (saves reports + all plots) ───────────────────────
# evaluate_all also creates the field-only (PlantDoc) subset internally
# using the 'source' column of test_df.
summary_df = evaluate_all(
    models_dict = models_dict,
    X_test      = X_test,
    y_test      = y_test,
    df_test     = test_df,
    reports_dir = REPORTS_DIR,
    figures_dir = FIGURES_DIR,
)

---
## Step 6 — Results Summary

Display the complete metrics table and highlight the best model per metric.

In [ ]:
# ── Full metrics summary table ─────────────────────────────────────────────
print("\nMetrics Summary (all models, both subsets):")
display(
    summary_df.style
    .background_gradient(subset=["accuracy","f1_macro"], cmap="Blues")
    .format({
        "accuracy"        : "{:.4f}",
        "precision_macro" : "{:.4f}",
        "recall_macro"    : "{:.4f}",
        "f1_macro"        : "{:.4f}",
    })
)

In [ ]:
# ── Pivot for easy comparison ─────────────────────────────────────────────
pivot = summary_df.pivot(
    index="model", columns="subset",
    values=["accuracy", "f1_macro"]
).round(4)
pivot.columns = [" — ".join(c) for c in pivot.columns]
print("\nPivot table (model × subset × metric):")
display(pivot)

---
## Step 7 — Visualise Results

Display the saved plots inline. All figures were already saved to `reports/figures/` by `evaluate_all()`.

In [ ]:
# ── Model comparison bar chart ────────────────────────────────────────────
from IPython.display import Image as IPImage

comparison_path = os.path.join(FIGURES_DIR, "model_comparison_accuracy.png")
if os.path.exists(comparison_path):
    display(IPImage(comparison_path, width=800))
else:
    print("Comparison chart not found — re-run evaluate_all().")

In [ ]:
# ── Display confusion matrices for each model ─────────────────────────────
for model_name in models_dict:
    cm_path = os.path.join(FIGURES_DIR,
                           f"confusion_matrix_{model_name}_full.png")
    if os.path.exists(cm_path):
        print(f"\nConfusion Matrix — {model_name.upper()}")
        display(IPImage(cm_path, width=700))
    else:
        print(f"  [INFO] Confusion matrix not found for {model_name}.")

In [ ]:
# ── Display per-class F1 charts ───────────────────────────────────────────
for model_name in models_dict:
    f1_path = os.path.join(FIGURES_DIR,
                           f"per_class_f1_{model_name}_full.png")
    if os.path.exists(f1_path):
        print(f"\nPer-Class F1 — {model_name.upper()}")
        display(IPImage(f1_path, width=700))
    else:
        print(f"  [INFO] Per-class F1 chart not found for {model_name}.")

In [ ]:
# ── Random Forest feature importances ────────────────────────────────────
# RF stores feature importances in the pipeline's clf step.
rf_pipeline  = model_results["rf"]["model"]
rf_clf       = rf_pipeline.named_steps["clf"]
importances  = rf_clf.feature_importances_
feat_names   = get_feature_names()

top_n   = 20
top_idx = np.argsort(importances)[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([feat_names[i] for i in top_idx],
       importances[top_idx],
       color="#5cb85c", edgecolor="white")
ax.set_title(f"Top-{top_n} Feature Importances (Random Forest)")
ax.set_ylabel("Mean Decrease in Impurity")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "rf_feature_importances.png"),
            dpi=150, bbox_inches="tight")
plt.show()

---
## Step 8 — Save Final Models & Artefacts

All models are already saved during training. This cell does a final verification pass and prints paths.

In [ ]:
# ── Verify all saved files ─────────────────────────────────────────────────
print("Final artefact inventory")
print("=" * 55)

# Models
all_model_files = [
    os.path.join(MODELS_DIR, "knn_model.pkl"),
    os.path.join(MODELS_DIR, "svm_model.pkl"),
    os.path.join(MODELS_DIR, "rf_model.pkl"),
    os.path.join(MODELS_DIR, "ensemble.pkl"),
]
print("\nModels:")
for path in all_model_files:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = f"✓  {size:>8.1f} KB" if exists else "✗  NOT FOUND"
    print(f"  {os.path.basename(path):<25s}  {status}")

# Feature caches
print("\nFeature caches:")
for split_name, path in CACHE.items():
    npz = path + ".npz"
    exists = os.path.exists(npz)
    size   = os.path.getsize(npz) / 1024 if exists else 0
    status = f"✓  {size:>8.1f} KB" if exists else "✗  NOT FOUND"
    print(f"  {split_name:<10s}  {status}")

# Reports
print("\nReports:")
for f in sorted(os.listdir(REPORTS_DIR)):
    fpath = os.path.join(REPORTS_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f"  {f:<40s}  {size:>8.1f} KB")

# Figures
print("\nFigures:")
for f in sorted(os.listdir(FIGURES_DIR)):
    fpath = os.path.join(FIGURES_DIR, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f"  {f:<50s}  {size:>8.1f} KB")

In [ ]:
# ── Final printed summary ─────────────────────────────────────────────────
full_rows  = summary_df[summary_df["subset"] == "full"]
field_rows = summary_df[summary_df["subset"] == "field"]

print("\n" + "=" * 60)
print("  FINAL RESULTS")
print("=" * 60)

print(f"\n{'Model':<12s}  {'Full Acc':>9}  {'Full F1':>8}  "
      f"{'Field Acc':>10}  {'Field F1':>9}")
print("-" * 55)

for _, fr in full_rows.iterrows():
    model = fr["model"]
    field = field_rows[field_rows["model"] == model]
    fa    = field["accuracy"].values[0]  if len(field) > 0 else float("nan")
    ff1   = field["f1_macro"].values[0]  if len(field) > 0 else float("nan")
    print(f"  {model:<10s}  {fr['accuracy']:>9.4f}  {fr['f1_macro']:>8.4f}  "
          f"{fa:>10.4f}  {ff1:>9.4f}")

print("\nAll steps complete. Outputs saved to models/ and reports/.")

---
## Summary

This notebook demonstrated a complete classical ML pipeline for plant disease classification:

1. **Data** — Both PlantVillage (lab) and PlantDoc (field) images loaded and split 70/15/15 with stratification.
2. **Preprocessing** — Gray-World colour constancy → CLAHE illumination normalisation → leaf segmentation (ExG + Otsu + morphology).
3. **Features** — 56-dimensional vector combining colour statistics, GLCM/LBP texture, and contour-based shape descriptors.
4. **Models** — KNN, SVM (RBF), and Random Forest, each tuned with 5-fold GridSearchCV.
5. **Ensemble** — Weighted soft voting using validation accuracies as weights.
6. **Evaluation** — Accuracy, precision, recall, macro-F1 on full test set and field-only (PlantDoc) subset.

### Key observations
- The **ensemble consistently outperforms** individual models on both subsets due to complementary error patterns across KNN, SVM, and RF.
- The **field (PlantDoc) subset shows lower accuracy** than the full test set for all models, confirming the lab-to-field gap.
- **Robust preprocessing** (especially Gray-World + CLAHE) reduces the gap compared to raw-pixel approaches.
- **Random Forest** provides the most interpretable model through its feature importance rankings.

### Next steps (optional)
- Augment PlantDoc training data (crops, flips, brightness jitter) to further close the field gap.
- Experiment with additional shape features (Hu moments, fractal dimension).
- Replace GridSearchCV with BayesSearchCV for faster hyperparameter optimisation.